# DNABERT2 Local/HPC Benchmark

This notebook runs the DNABERT2 frozen embedding benchmark in a local Linux/HPC-style environment. It uses the same benchmark contract as CNN-v2:

- same train/validation/test CSV files
- seed `42`
- validation-only threshold selection using MCC
- test set used only for final reporting
- primary comparison metric: test MCC
- secondary comparison metric: test AUPRC
- shared artifacts: `metrics.csv`, `metrics.json`, `predictions.csv`, `manifest.json`, `history.csv`, `checkpoints/`, and `embeddings/`

Use this notebook when Colab fails because of Triton, CUDA memory, or DNABERT2 remote-code compatibility. For the full dataset, a Linux GPU node or supercomputer job is preferred.

## 1. Environment recommendation

DNABERT2 uses custom Hugging Face remote code. The original SeqTrainer DNABERT notebooks used an older Transformers stack where missing Triton could fall back to PyTorch attention. Newer Colab stacks can instead crash with `tl.dot(..., trans_b=True)`.

Recommended environment for HPC:

```bash
module load cuda  # use the CUDA module available on your cluster
python -m venv .venv-dnabert2
source .venv-dnabert2/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install -e ".[torch]"
python -m pip install "transformers==4.29.2" "einops>=0.6" "accelerate>=0.20"
```

Do not tune the threshold or hyperparameters on the test split. Keep all model comparisons on the same CSV split files.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

# If this notebook is opened from inside the repo, keep this as Path.cwd().
# Otherwise set REPO_DIR manually, for example:
# REPO_DIR = Path('/path/to/SeqTrainer')
REPO_DIR = Path.cwd()
if not (REPO_DIR / 'src' / 'seqtrainer').exists():
    for candidate in [REPO_DIR, *REPO_DIR.parents]:
        if (candidate / 'src' / 'seqtrainer').exists():
            REPO_DIR = candidate
            break
    else:
        raise RuntimeError(f"REPO_DIR does not look like SeqTrainer: {REPO_DIR}")

os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Repository:', REPO_DIR)
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], check=False)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=False)


## 2. Install/check dependencies

Run this once in a fresh local/HPC environment. On a cluster, install PyTorch according to the cluster CUDA instructions if the default extra does not match the node CUDA version.

In [ ]:
# Local/HPC dependency check.
# If imports fail, uncomment the install lines below in your environment.
# %pip install -e ".[torch]"
# %pip install "transformers==4.29.2" "einops>=0.6" "accelerate>=0.20"

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
import einops

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('einops:', einops.__version__)


## Original DNABERT notebook pattern kept here

The older `dna-bert/Colab_test_dnabert2.ipynb` did three things that are still useful:

1. load `AutoTokenizer`, `AutoConfig`, and `AutoModel` first as a small sanity check;
2. use explicit `model_max_length`, train/eval/test directories, and GPU batch sizes;
3. run the heavy model through a Linux/conda/`torchrun` environment, not through fragile notebook-only state.

This notebook keeps those ideas, but routes the real benchmark through SeqTrainer so the outputs match CNN-v2: same split files, same metrics, same validation-only threshold policy, and same artifact layout.

In [ ]:
# DNABERT2 lightweight load sanity check inspired by the original repo notebooks.
# This checks model/tokenizer compatibility before launching the full benchmark.
# If this fails because of Triton on Windows/Colab, use the HPC/SLURM path below.

from transformers import AutoConfig, AutoModel, AutoTokenizer

MODEL_NAME = 'zhihan1996/DNABERT-2-117M'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)

if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

print('tokenizer pad_token_id:', tokenizer.pad_token_id)
print('config pad_token_id:', getattr(config, 'pad_token_id', None))
print('hidden_size:', getattr(config, 'hidden_size', None))
print('This sanity cell only loads config/tokenizer. The full encoder loads in the benchmark cell.')


## 3. Verify shared split files

These are the same three CSV files used by the CNN benchmark. Keeping these fixed is what makes CNN-v2 and DNABERT2 comparable.

In [ ]:
DATA_DIR = REPO_DIR / 'data' / 'promoter_classification'
SPLITS = {
    'train': DATA_DIR / 'train_EP_DNA_BERT2_genomic_order.csv',
    'validation': DATA_DIR / 'eval_EP_DNA_BERT2_genomic_order.csv',
    'test': DATA_DIR / 'test_EP_DNA_BERT2_genomic_order.csv',
}

missing = [str(path) for path in SPLITS.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing shared split CSVs:' + chr(10) + chr(10).join(missing))

for split, path in SPLITS.items():
    frame = pd.read_csv(path)
    print(split, path)
    print('rows:', len(frame))
    print('columns:', list(frame.columns))
    print('labels:', frame['label'].value_counts(dropna=False).to_dict())
    print()


## 4. Benchmark config constants

This config must stay aligned with CNN-v2 except for model family/preprocessing. The shared constants are split files, seed, label policy, threshold policy, and metrics.

Important DNABERT2 setting: `model_max_length = 104` is divisible by `pad_to_multiple_of = 8`, preventing the tokenizer error caused by `100` and `8` being incompatible.

In [ ]:
CONFIG = REPO_DIR / 'config-examples' / 'benchmarks' / 'dnabert2_frozen.toml'
if not CONFIG.exists():
    raise FileNotFoundError(CONFIG)

print(CONFIG)
print(CONFIG.read_text(encoding='utf-8'))


## 5. Optional tokenization smoke check

This does not train the model. It verifies that DNABERT2 tokenization can read the same split files before downloading/loading the full encoder.

In [ ]:
# Optional. Useful before submitting a long HPC job.
!python -m seqtrainer.cli.main benchmark prepare-dnabert2 {CONFIG} --base-dir {REPO_DIR} --output-dir outputs/benchmarks/dnabert2_tokenization_check


## 6. Run frozen DNABERT2 benchmark

This is the main benchmark. It extracts DNABERT2 embeddings in batches, trains only the classifier head, chooses the threshold on validation MCC, and evaluates test once using that validation-selected threshold.

If the environment cannot load DNABERT2, the benchmark should fail clearly or write a skipped manifest when `allow_skip=True`. Do not fake metrics.

In [ ]:
from seqtrainer.benchmarks.runner import run_benchmark

# This machine has only Intel integrated graphics, so CUDA is not available here.
# To still verify the full artifact contract locally, the notebook runs a small,
# stratified CPU smoke benchmark when CUDA is absent. The CPU smoke metrics are
# real metrics, but they are NOT the final scientific comparison because they use
# only a small subset. Use HPC/GPU for the full shared-split DNABERT2 metrics.
RUN_CPU_SMOKE_WHEN_NO_CUDA = True
CPU_SMOKE_ROWS = {"train": 16, "validation": 8, "test": 8}
FORCE_CPU_FULL_BENCHMARK = False


def write_cpu_smoke_inputs_and_config():
    smoke_data_dir = REPO_DIR / "data" / "promoter_classification_cpu_smoke"
    smoke_data_dir.mkdir(parents=True, exist_ok=True)

    source_paths = {
        "train": REPO_DIR / "data" / "promoter_classification" / "train_EP_DNA_BERT2_genomic_order.csv",
        "validation": REPO_DIR / "data" / "promoter_classification" / "eval_EP_DNA_BERT2_genomic_order.csv",
        "test": REPO_DIR / "data" / "promoter_classification" / "test_EP_DNA_BERT2_genomic_order.csv",
    }

    for split, source_path in source_paths.items():
        frame = pd.read_csv(source_path)
        pieces = []
        for _, group in frame.groupby("label"):
            per_class = max(1, CPU_SMOKE_ROWS[split] // 2)
            pieces.append(group.sample(n=min(len(group), per_class), random_state=42))
        sample = pd.concat(pieces).sample(frac=1, random_state=42).reset_index(drop=True)
        sample.to_csv(smoke_data_dir / f"{split}.csv", index=False)
        print(f"CPU smoke {split}: {len(sample)} rows -> {smoke_data_dir / f'{split}.csv'}")

    smoke_config = REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_cpu_smoke.toml"
    smoke_toml = '''[experiment]
name = "dnabert2_cpu_smoke_ep_genomic_order"
task = "bacterial_promoter_prediction"
description = "CPU smoke DNABERT2 run on a small stratified subset for artifact generation only."
seed = 42

[dataset]
name = "ep_dnabert2_genomic_order_cpu_smoke"
format = "csv"
source_accession = "GSE144621"
source_url = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE144621"
version = "EP_DNA_BERT2_genomic_order_cpu_smoke"
sequence_field = "sequence"
label_field = "label"

[dataset.split_files]
train = "data/promoter_classification_cpu_smoke/train.csv"
validation = "data/promoter_classification_cpu_smoke/validation.csv"
test = "data/promoter_classification_cpu_smoke/test.csv"

[label]
source = "provided_binary"
positive_label = 1
negative_label = 0

[split]
strategy = "predefined"
seed = 42
validation_name = "eval"

[preprocessing]
encoding = "dnabert2_tokenizer"
sequence_length = 300
pad_or_trim = false

[preprocessing.params]
model_max_length = 104
padding = "longest"
pad_to_multiple_of = 8

[model]
family = "dnabert2"
name = "zhihan1996/DNABERT-2-117M"
version = "pin_model_revision_before_final_report"

[model.params]
mode = "frozen_embedding_classifier"
pooling = "mean"
trust_remote_code = true
allow_download = true
require_model_files = true
classifier_dropout = 0.1
cache_embeddings = true
disable_flash_attention = true

[training]
seed = 42
batch_size = 1
max_epochs = 3
learning_rate = 0.0003

[training.params]
optimizer = "adamw"
loss = "bce_with_logits"
warmup_ratio = 0.0
weight_decay = 0.0001
early_stopping_patience = 2

[evaluation]
primary_metric = "mcc"
threshold_strategy = "validation_mcc"
metrics = ["accuracy", "balanced_accuracy", "auroc", "auprc", "f1", "mcc", "precision", "sensitivity", "specificity", "confusion_matrix"]

[outputs]
output_dir = "outputs/benchmarks/dnabert2_cpu_smoke_ep_genomic_order"
save_json = true
save_csv = true
save_predictions = true

[environment]
device = "cpu"
precision = "float32"
'''
    smoke_config.write_text(smoke_toml, encoding="utf-8")
    return smoke_config


if torch.cuda.is_available():
    ACTIVE_CONFIG = CONFIG
    print("CUDA is visible. Running the full DNABERT2 shared-split benchmark.")
elif FORCE_CPU_FULL_BENCHMARK:
    ACTIVE_CONFIG = CONFIG
    print("CUDA is not visible. FORCE_CPU_FULL_BENCHMARK=True, so this may be very slow.")
elif RUN_CPU_SMOKE_WHEN_NO_CUDA:
    ACTIVE_CONFIG = write_cpu_smoke_inputs_and_config()
    print("CUDA is not visible. Running a small CPU smoke benchmark so metrics/artifacts are produced locally.")
    print("Use the full config on HPC/GPU for final comparable DNABERT2 results.")
else:
    ACTIVE_CONFIG = None
    print("CUDA is not visible and CPU smoke mode is disabled.")

if ACTIVE_CONFIG is None:
    result = None
else:
    result = run_benchmark(ACTIVE_CONFIG, base_dir=REPO_DIR, allow_skip=False)
    print("status:", result.status)
    print("output_dir:", result.output_dir)
    print(json.dumps(result.metrics, indent=2))


## 7. Inspect artifacts

The output directory should contain the shared benchmark artifacts. Frozen DNABERT2 should also contain cached embeddings.

In [ ]:
if result is not None:
    OUTPUT_DIR = Path(result.output_dir)
    if not OUTPUT_DIR.is_absolute():
        OUTPUT_DIR = REPO_DIR / OUTPUT_DIR
else:
    OUTPUT_DIR = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_frozen_ep_genomic_order"

metrics_path = OUTPUT_DIR / "metrics.csv"
if metrics_path.exists():
    print("Output directory:", OUTPUT_DIR)
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            try:
                print(path.relative_to(REPO_DIR))
            except ValueError:
                print(path)

    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print("No complete DNABERT2 metrics.csv yet.")
    print("Output directory checked:", OUTPUT_DIR)


## 8. Compare against CNN-v2

Run this after both CNN-v2 and DNABERT2 have completed. The comparison ranks primarily by held-out test MCC and secondarily by held-out test AUPRC. The threshold is still selected only on validation for each model.

In [ ]:
CNN_V2_OUT = REPO_DIR / 'outputs' / 'benchmarks' / 'cnn_v2_regularized_ep_genomic_order'
DNABERT2_OUT = OUTPUT_DIR
COMPARE_OUT = REPO_DIR / 'outputs' / 'benchmarks' / 'comparison_cnn_v2_dnabert2'

if CNN_V2_OUT.exists() and DNABERT2_OUT.exists():
    !python -m seqtrainer.cli.main benchmark compare {CNN_V2_OUT} {DNABERT2_OUT} --output-dir {COMPARE_OUT}
    display(pd.read_csv(COMPARE_OUT / 'comparison_metrics.csv'))
    print((COMPARE_OUT / 'comparison_summary.md').read_text(encoding='utf-8'))
else:
    print('Run CNN-v2 and DNABERT2 first before comparison.')
    print('Missing CNN-v2 output:', not CNN_V2_OUT.exists())
    print('Missing DNABERT2 output:', not DNABERT2_OUT.exists())


## Optional: Trainer-style fine-tuning reference

The original DNABERT notebook used `torchrun` and `train.py` with arguments such as `--model_max_length`, `--per_device_train_batch_size`, `--learning_rate`, warmup, logging, and best-model loading.

For SeqTrainer, keep this as a secondary path. The primary comparable benchmark is still `seqtrainer benchmark run ...` because it writes the same metrics and predictions as CNN-v2. Use Trainer-style fine-tuning only after the frozen baseline runs cleanly.

In [ ]:
# Optional fine-tuning benchmark through the SeqTrainer config, not ad hoc metrics.
# Run this only on a Linux GPU/HPC node after the frozen benchmark is healthy.

FINETUNE_CONFIG = REPO_DIR / 'config-examples' / 'benchmarks' / 'dnabert2_finetune.toml'
print(FINETUNE_CONFIG)

# Uncomment on HPC when ready:
# !python -m seqtrainer.cli.main benchmark run {FINETUNE_CONFIG} --base-dir {REPO_DIR} --strict


Trainer-style settings from the original DNABERT notebooks that map to our configs:

- `model_max_length`: keep compatible with `pad_to_multiple_of`; this notebook uses `104`.
- `learning_rate`: frozen head can use higher LR such as `1e-3`; full/partial fine-tuning should use lower LR such as `3e-5`.
- `warmup_steps`/`warmup_ratio`: keep warmup enabled for fine-tuning stability.
- `load_best_model_at_end`/best metric: in SeqTrainer this is represented by best validation MCC selection.
- `fp16`/`bf16`: use only on supported Linux GPU nodes.

Do not copy the old regression metric cells into this benchmark, because this task is binary promoter classification.

## 9. Example SLURM command

Use the same config and command outside the notebook for the real supercomputer run.

```bash
#!/bin/bash
#SBATCH --job-name=seqtrainer-dnabert2
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --time=12:00:00

source .venv-dnabert2/bin/activate
cd /path/to/SeqTrainer
export PYTHONPATH=src
python -m seqtrainer.cli.main benchmark run config-examples/benchmarks/dnabert2_frozen.toml --base-dir . --strict
```

Keep the run artifacts and commit hash for the report.